# Eight Gaussians OFM with an amortized inverse warm start

A small MLP takes `(x_t, t)` and predicts a residual correction `z_init = x_t + t * MLP(x_t, t)`. The zero initialization of its final layer starts the experiment at `z_init = x_t`. The original capped L-BFGS then refines this warm start using its default step rule.

A separate Adam optimizer regresses the prediction onto the detached L-BFGS solution, following OFM Appendix B.6, Algorithm 2 and the regression-based amortization approach of Amos. A capped result supplies a regression target only when its inverse residual norm is at or below `INV_TOLERANCE`. The profile records the fraction of valid targets and the MLP time. Total iteration time includes the prediction and the MLP update.

[OFM Appendix B.6](https://arxiv.org/html/2403.13117#A2.SS6) · [Amos, On amortizing convex conjugates for optimal transport](https://arxiv.org/html/2210.12153v3)

In [ ]:
# parameters
DEVICE='mps'
MODEL = 'v2cpf'
INV_MAX_ITER = 10  # cap on outer L-BFGS steps of the inner solve
AMORT_WIDTH = 64
AMORT_LR = 1e-3

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import torch
import torch.distributions as TD
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import clear_output
import random
import time

import sys, os
from pathlib import Path

notebook_dir = Path.cwd()
if (notebook_dir / 'ofm_notebooks').is_dir():
    notebook_dir = notebook_dir / 'ofm_notebooks'
sys.path.append(str(notebook_dir.parent))

# icnns
from ofmsrc.icnn import (
    ICNNCPF,
    ICNN2CPF,
    ICNN3CPF,
    LseICNNCPF,
    ResICNN2CPF,
    DenseICNN2CPF
)
from ofmsrc.icnn import (
    LinActnormICNN,
    DenseICNN
)

from ofmsrc.model_tools import (
    id_pretrain_model,
    ofm_forward,
    ofm_loss
)
from ofmsrc.model_tools import (
    TorchoptimInvParams,
    TorchlbfgsInvParams,
    BruteforceIHVParams
)

# dgm_utils
from ofmsrc.dgm_utils import visualize_2d_samples_groups, visualize_2d_map
from ofmsrc.tools import freeze, unfreeze, EMA

# spiral dataset etc.
from ofmsrc.distributions import Sampler, StandardNormalSampler
from dataclasses import replace

In [ ]:
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.random.manual_seed(seed)
    random.seed(seed)

### Source and Target distributions

In [ ]:
class Mix8GaussiansSampler(Sampler):
    def __init__(
        self, with_central=False, std=1, r=12, dim=2, device='cuda'
    ):
        super().__init__(
            device=device
        )
        assert dim == 2
        self.dim = 2
        self.std, self.r = std, r
        
        self.with_central = with_central
        centers = [
            (1, 0), (-1, 0), (0, 1), (0, -1),
            (1. / np.sqrt(2), 1. / np.sqrt(2)),
            (1. / np.sqrt(2), -1. / np.sqrt(2)),
            (-1. / np.sqrt(2), 1. / np.sqrt(2)),
            (-1. / np.sqrt(2), -1. / np.sqrt(2))
        ]
        if self.with_central:
            centers.append((0, 0))
        self.centers = torch.tensor(
            centers, device=self.device
        )
        
    def sample(self, batch_size=10):
        batch = torch.randn(
            batch_size, self.dim,
            device=self.device
        )
        indices = random.choices(range(len(self.centers)), k=batch_size)
        batch *= self.std
        batch += self.r * self.centers[indices, :]
        return torch.tensor(
            batch, device=self.device
        )

seed_everything()

mix8gau_sampler = Mix8GaussiansSampler(device='cpu', r=3, std=0.4)
standardnormal_sampler = StandardNormalSampler(device='cpu')

source_sampler = standardnormal_sampler
target_sampler = mix8gau_sampler

visualize_2d_samples_groups([
    source_sampler.sample(5024).cpu().numpy(), 
    target_sampler.sample(5024).cpu().numpy()], 'Source&Target', s=3)

### Model

In [ ]:
seed_everything()

sampler = TD.Normal(
    torch.tensor([0., 0.]).to(DEVICE), 
    torch.tensor([4., 4.]).to(DEVICE))

def get_model(name):
    assert name in [
        'v2cpf'
    ]
    print(f"Used model: {name}")
    model = ICNN2CPF(dim=2, dimh=1024, num_hidden_layers=2, batch_size=1024).to(DEVICE)
    id_pretrain_model(model, sampler, n_max_iterations=200, lr=1e-2)
    return model


In [ ]:
model = get_model(MODEL).to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=1e-2)

# Preserve the training RNG stream when initializing the separate CPU MLP.
with torch.random.fork_rng(devices=[]):
    torch.random.default_generator.manual_seed(43)
    amortizer = torch.nn.Sequential(
        torch.nn.Linear(3, AMORT_WIDTH),
        torch.nn.SiLU(),
        torch.nn.Linear(AMORT_WIDTH, AMORT_WIDTH),
        torch.nn.SiLU(),
        torch.nn.Linear(AMORT_WIDTH, 2),
    )
    torch.nn.init.zeros_(amortizer[-1].weight)
    torch.nn.init.zeros_(amortizer[-1].bias)
amortizer = amortizer.to(DEVICE)
amort_optim = torch.optim.Adam(amortizer.parameters(), lr=AMORT_LR)

### Output directory

In [ ]:
EXP_DIR = notebook_dir / f'gau_{MODEL}_cap{INV_MAX_ITER}_amort_mlp{AMORT_WIDTH}'
EXP_DIR.mkdir(exist_ok=True)

samples_path = EXP_DIR / 'samples.png'
curr_model_path = EXP_DIR / 'model.pth'
loss_path = EXP_DIR / 'loss.png'

### Fitting eight gaussians

In [ ]:
MAX_ITER = 30000  # total target; raise it and re-run to resume from the checkpoint
CKPT_EVERY = 500
BATCH_SIZE = 1024
INV_TOLERANCE = 0.01

inverse_params = TorchlbfgsInvParams(
    lbfgs_params=dict(tolerance_grad=INV_TOLERANCE),
    max_iter=INV_MAX_ITER,
    record_steps=True,
    report_final_norms=True,
    report_solution=True,
)


In [ ]:
ema = EMA(0, betas=[0.999, 0.99])

In [ ]:
import pandas as pd
from copy import deepcopy

CKPT_PATH = os.path.join(EXP_DIR, 'ckpt.pt')
PROFILE_CSV = os.path.join(EXP_DIR, 'profile.csv')
PROFILE_PNG = os.path.join(EXP_DIR, 'profile.png')

# per-iteration history (one entry per training iteration, times in seconds)
HIST_KEYS = ['loss', 'true_loss', 't_total', 't_inverse', 't_ihv', 'inv_steps', 'inv_norms_max', 't_min', 't_diag']
HIST_KEYS += ['t_max', 'inv_func_evals', 'inv_initial_norm_max']
HIST_KEYS += ['amort_loss', 'amort_target_fraction', 't_amort']
hist = {k: [] for k in HIST_KEYS}

# per-iteration convergence curves of the inner solve: one row per iteration, one column per outer L-BFGS step
# (column 0 = the initial point), NaN after the solve stopped
#   max_abs: max |g| over the batch (what tolerance_grad is compared against)
#   norm_max / norm_mean: max / mean over the batch of the per-sample l2 norm of g
#   n_iter / func_evals: cumulative inner L-BFGS iterations / closure evaluations
CONV_KEYS = ['max_abs', 'norm_max', 'norm_mean', 'n_iter', 'func_evals']
conv = {k: [] for k in CONV_KEYS}
CONV_NPZ = os.path.join(EXP_DIR, 'conv.npz')
CONV_PNG = os.path.join(EXP_DIR, 'conv.png')
start_iter = 0


def plot_profile(w=100):
    def smooth(a):
        a = np.asarray(a, dtype=float)
        if len(a) < w:
            return np.arange(len(a)), a
        return np.arange(w - 1, len(a)), np.convolve(a, np.ones(w) / w, mode='valid')

    fig, ax = plt.subplots(2, 2, figsize=(14, 8))
    ax[0, 0].semilogy(hist['true_loss'], lw=0.5); ax[0, 0].set_title('true_loss')
    for k in ['t_total', 't_inverse', 't_ihv', 't_amort']:
        x, y = smooth(hist[k])
        ax[0, 1].plot(x, 1e3 * y, label=k)
    ax[0, 1].set_title(f'time per iter, ms (moving avg {w})'); ax[0, 1].legend()
    x, y = smooth(hist['inv_steps'])
    ax[1, 0].plot(x, y); ax[1, 0].set_title(f'L-BFGS steps (moving avg {w})')
    ax[1, 1].semilogy(hist['inv_norms_max'], lw=0.5); ax[1, 1].set_title('inverse max grad norm')
    for a in ax.flat:
        a.set_xlabel('iteration')
    fig.tight_layout(); fig.savefig(PROFILE_PNG); plt.close(fig)


def plot_conv(tol=None):
    A = {k: np.asarray(v) for k, v in conv.items()}  # (N, INV_MAX_ITER + 1)
    N = len(A['max_abs'])

    def locf(a):  # solves that stopped early stay at their last residual
        a = a.copy()
        for j in range(1, a.shape[1]):
            m = np.isnan(a[:, j])
            a[m, j] = a[m, j - 1]
        return a

    starts = sorted({s for s in [0, 10000, 20000, N - 1000] if 0 <= s < N})
    windows = [(s, min(s + 1000, N)) for s in starts]
    steps = np.arange(A['max_abs'].shape[1])
    fig, ax = plt.subplots(2, 2, figsize=(14, 8))
    for lo, hi in windows:
        lab = f'iters {lo}-{hi}'
        L = {k: locf(A[k][lo:hi]) for k in ['max_abs', 'norm_max', 'norm_mean']}
        line, = ax[0, 0].semilogy(steps, np.median(L['max_abs'], 0), label=lab)
        ax[0, 0].semilogy(steps, np.quantile(L['max_abs'], 0.9, axis=0), '--', color=line.get_color())
        ax[0, 1].semilogy(steps, np.median(L['norm_max'], 0), color=line.get_color(), label=lab)
        ax[1, 0].semilogy(steps, np.median(L['norm_mean'], 0), color=line.get_color(), label=lab)
        ax[1, 1].plot(steps, (~np.isnan(A['max_abs'][lo:hi])).mean(0), color=line.get_color(), label=lab)
    if tol is not None:
        ax[0, 0].axhline(tol, color='k', ls=':', lw=1)
    ax[0, 0].set_title('max |g| over batch (stopping criterion): median, dashed = p90')
    ax[0, 1].set_title('max per-sample l2 norm of g: median')
    ax[1, 0].set_title('mean per-sample l2 norm of g: median')
    ax[1, 1].set_title('fraction of solves that performed >= k outer steps')
    for a in ax.flat:
        a.set_xlabel('outer L-BFGS step'); a.legend()
    fig.tight_layout(); fig.savefig(CONV_PNG); plt.close(fig)


def save_checkpoint():
    n = len(hist['loss'])
    if n == 0:
        return
    ckpt = dict(
        iteration=n,
        model=model.state_dict(),
        optim=optim.state_dict(),
        amortizer=amortizer.state_dict(),
        amort_optim=amort_optim.state_dict(),
        ema_step=ema._step,
        ema_models=[m.state_dict() for m in ema.model_copies],
        hist=hist,
        conv=conv,
        rng=dict(
            numpy=np.random.get_state(), torch=torch.get_rng_state(),
            cuda=torch.cuda.get_rng_state_all(), python=random.getstate(),
            mps=torch.mps.get_rng_state() if DEVICE == 'mps' else None
        )
    )
    torch.save(ckpt, CKPT_PATH + '.tmp')
    os.replace(CKPT_PATH + '.tmp', CKPT_PATH)  # atomic: an interrupted save can't corrupt the checkpoint
    torch.save(model.state_dict(), curr_model_path)
    for beta in [0.999, 0.99]:
        beta_model_path = os.path.join(EXP_DIR, 'model_{}.pth'.format(beta))
        torch.save(ema.get_model(beta), beta_model_path)
    df = pd.DataFrame(hist)
    df.index.name = 'iteration'
    df.to_csv(PROFILE_CSV)
    np.savez_compressed(CONV_NPZ, **{k: np.asarray(v, dtype=np.float32) for k, v in conv.items()})
    plot_profile()
    plot_conv(INV_TOLERANCE)


# resume from the checkpoint if there is one
if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optim.load_state_dict(ckpt['optim'])
    amortizer.load_state_dict(ckpt['amortizer'])
    amort_optim.load_state_dict(ckpt['amort_optim'])
    ema.model_copies = [deepcopy(model) for _ in ema.betas]
    for m, sd in zip(ema.model_copies, ckpt['ema_models']):
        m.load_state_dict(sd)
    ema._step = ckpt['ema_step']
    hist, conv, start_iter = ckpt['hist'], ckpt['conv'], ckpt['iteration']
    np.random.set_state(ckpt['rng']['numpy'])
    torch.set_rng_state(ckpt['rng']['torch'].cpu())
    torch.cuda.set_rng_state_all([s.cpu() for s in ckpt['rng']['cuda']])
    if DEVICE == 'mps' and ckpt['rng'].get('mps') is not None:
        torch.mps.set_rng_state(ckpt['rng']['mps'].cpu())
    random.setstate(ckpt['rng']['python'])
    print(f'Resumed from {CKPT_PATH} at iteration {start_iter}')


In [ ]:
print('Fitting the eight Gaussians')
if start_iter >= MAX_ITER:
    print(f'Already at iteration {start_iter} >= MAX_ITER={MAX_ITER}; raise MAX_ITER to continue training.')

try:
    with tqdm(range(start_iter, MAX_ITER)) as tbar:
        for iteration in tbar:
            t_iter = time.perf_counter()
            X = source_sampler.sample(BATCH_SIZE).to(DEVICE)
            Y = target_sampler.sample(BATCH_SIZE).to(DEVICE)
            t = (torch.rand(BATCH_SIZE) + 1e-8).to(DEVICE)

            amort_started = time.perf_counter()
            t_col = t[:, None]
            Xt = X * (1 - t_col) + Y * t_col
            z_init = Xt + t_col * amortizer(torch.cat((Xt, t_col), dim=1))
            batch_inverse_params = replace(inverse_params, initial_guess=z_init.detach())
            t_amort = time.perf_counter() - amort_started

            current_metrics = dict()
            loss, true_loss = ofm_loss(
                model, X, Y, t, batch_inverse_params, BruteforceIHVParams(), stats=current_metrics
            )

            optim.zero_grad()
            loss.backward()
            optim.step(); model.convexify();
            current_metrics['loss'] = loss.item(); current_metrics['true_loss'] = true_loss.item()

            ema(model)

            # profiling: time spent in the inner inverse solve vs. the rest (the time of the
            # per-step diagnostics, t_diag, is excluded from t_total and t_inverse)
            inv_stats = current_metrics['inverse']
            amort_started = time.perf_counter()
            solved = t >= inverse_params.t_min
            # The solver reports residuals in the order of the rows it actually solved.
            valid_target = inv_stats['norms'] <= INV_TOLERANCE
            n_targets = int(valid_target.sum().item())
            amort_loss = float('nan')
            if n_targets:
                target = inv_stats['solution'][solved][valid_target]
                error = z_init[solved][valid_target] - target
                regression_loss = error.square().sum(dim=1).mean()
                amort_optim.zero_grad(set_to_none=True)
                regression_loss.backward()
                amort_optim.step()
                amort_loss = regression_loss.item()
            t_amort += time.perf_counter() - amort_started
            hist['amort_loss'].append(amort_loss)
            hist['amort_target_fraction'].append(n_targets / len(t))
            hist['t_amort'].append(t_amort)

            t_diag = inv_stats['time_diag']
            pad = lambda a: np.concatenate([a, np.full(INV_MAX_ITER + 1 - len(a), np.nan)])
            ss = inv_stats['step_stats']
            conv['max_abs'].append(pad(ss[:, 0]))
            conv['norm_max'].append(pad(ss[:, 1]))
            conv['norm_mean'].append(pad(ss[:, 2]))
            conv['n_iter'].append(pad(np.asarray(inv_stats['step_n_iter'], dtype=float)))
            conv['func_evals'].append(pad(np.asarray(inv_stats['step_func_evals'], dtype=float)))
            hist['loss'].append(current_metrics['loss'])
            hist['true_loss'].append(current_metrics['true_loss'])
            hist['t_inverse'].append(current_metrics['times']['inverse'] - t_diag)
            hist['t_ihv'].append(current_metrics['times']['ihv'])
            hist['inv_steps'].append(current_metrics['inverse'].get('steps', float('nan')))
            hist['inv_norms_max'].append(current_metrics['inverse'].get('norms_max', float('nan')))
            hist['t_min'].append(t.min().item())
            hist['t_max'].append(t.max().item())
            hist['inv_func_evals'].append(inv_stats['step_func_evals'][-1])
            hist['inv_initial_norm_max'].append(float(ss[0, 1]))
            hist['t_diag'].append(t_diag)
            hist['t_total'].append(time.perf_counter() - t_iter - t_diag)

            tbar.set_postfix(loss=current_metrics['loss'], tloss = current_metrics['true_loss'],
                             inv_frac=np.mean(hist['t_inverse'][-100:]) / np.mean(hist['t_total'][-100:]))
            if iteration % 500 == 0 or iteration == MAX_ITER - 1:
                clear_output(wait=True)
                print("Iteration", iteration)
                print(f'\nLoss: {true_loss.item()}')
                avg = lambda k: 1e3 * np.mean(hist[k][-500:])
                print(f"Profile (avg last 500 iters): total {avg('t_total'):.1f} ms | inverse {avg('t_inverse'):.1f} ms "
                      f"({100*avg('t_inverse')/avg('t_total'):.0f}%) | ihv {avg('t_ihv'):.1f} ms | "
                      f"L-BFGS steps {np.mean(hist['inv_steps'][-500:]):.1f} | cap hit {100*np.mean(np.asarray(hist['inv_steps'][-500:]) >= INV_MAX_ITER):.0f}%")
                plt.plot(hist['true_loss'])
                plt.savefig(loss_path)
                plt.show()
                X = source_sampler.sample(5000).to(DEVICE)
                Y = model.push_nograd(X)
                visualize_2d_map(
                    X.cpu().numpy(),
                    Y.cpu().numpy(),
                    'Learned mapping',
                    data_label='src', mapped_data_label='mpd', s=3, figsize=(8, 8), save_path=samples_path)
            if (iteration + 1) % CKPT_EVERY == 0:
                save_checkpoint()
except KeyboardInterrupt:
    print('Interrupted; saving checkpoint')

if len(hist['loss']) > start_iter:
    save_checkpoint()
    print(f"Checkpoint saved at iteration {len(hist['loss'])}: {CKPT_PATH}")
